Project: /data-manager/api/_project.yaml
Book: /data-manager/api/_book.yaml

<style>
  devsite-code .tfo-notebook-code-cell-output {
    max-height: 300px;
    overflow: auto;
    background: rgba(255, 247, 237, 1);  /* light orange bg */
  }
  
  devsite-code .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(255, 247, 237, .7);
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output {
    background: rgba(64, 78, 103, 1);  /* dark mode slate */
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(64, 78, 103, .7);
  }
  
  .devsite-table-wrapper .tfo-notebook-buttons {
    display: inline-block;
    margin-left: 3px;
    width: auto;
    border: 0;
  }
  
  .tfo-notebook-buttons tr {
    background: 0;
    border: 0;
  }
  
  .tfo-notebook-buttons td {
    padding-left: 0;
    padding-right: 20px;
    border: 0;
  }
  
  .tfo-notebook-buttons {
    --tfo-notebook-buttons-box-shadow: 0 1px 2px 0 rgba(60, 64, 67, .3), 0 1px 3px 1px rgba(60, 64, 67, .15);
  }
  
  .tfo-notebook-buttons a,
  .tfo-notebook-buttons :link,
  .tfo-notebook-buttons :visited {
    border-radius: 8px;
    box-shadow: var(--tfo-notebook-buttons-box-shadow);
    color: #202124;
    padding: 12px 24px;
    transition: box-shadow 0.2s;
    text-decoration: none;
    display: flex;
    align-items: center;
  }
  
  .tfo-notebook-buttons a:hover,
  .tfo-notebook-buttons a:focus {
    box-shadow: 0 2px 6px 2px rgba(60, 64, 67, 0.15);
    text-decoration: none;
  }
  
  .tfo-notebook-buttons td > a > img {
    margin-right: 8px;
    width: 32px;
    height: 32px;
  }
  </style>

In [ ]:
# @markdown #### Copyright 2026 Google LLC
# @markdown ##### Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Data Partner: Dual Persona Workflow (In-Platform Experience)

  <table class="tfo-notebook-buttons nocontent" align="left">
    <td>
      <a target="_blank" href="https://colab.research.google.com/github/googleads/data-manager-python/blob/main/notebooks/audience_e2e_data_partner_dual_persona.ipynb">
      <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
      Run in Google Colab</a>
    </td>
    <td>
      <a target="_blank" href="https://github.com/googleads/data-manager-python/blob/main/notebooks/audience_e2e_data_partner_dual_persona.ipynb">
      <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
      View source on GitHub</a>
    </td>
  </table>

## Objective

This notebook simulates the exact flow required for App Verification, where advertisers link to a partner within your UI.

## Prerequisites

1. The JSON for a **Desktop app** OAuth client.

   [Follow these instructions](https://developers.google.com/data-manager/api/devguides/quickstart/set-up-access#user-account) if you don't have this JSON file. You'll use the JSON file you download for the prompt in **Phase 0**.

2. A Google Account with the **Admin**
   [access level](https://support.google.com/google-ads/answer/9978556) in the Advertiser account.

   The notebook will prompt you to login as this account during Phase 1 unless you provide existing
   credentials (a client ID, client secret, and refresh token) in Phase 0.
   
3. The email address and project ID of a [service account](//cloud.google.com/docs/authentication#service-accounts)
   with the **Standard** [access level](https://support.google.com/google-ads/answer/9978556) in the Data Partner account.

   [Follow these instructions](https://developers.google.com/data-manager/api/devguides/quickstart/set-up-access#service-account) if you don't have a service account.


## Phases

* **Phase 0:** Provide OAuth Information and Account IDs

  Fill in the prompts with information about your service account, data partner, and advertiser account.

  Upload the JSON file for your OAuth Desktop app.

* **Phase 1:** Act as the Advertiser

  Create a partner link from the advertiser to the data partner. This step uses user credentials for a user with access to the advertiser account.

* **Phase 2:** Act as the Data Partner

  Retrieve the partner link, create an audience, push data, and check the processing status. This step uses service account credentials for a service account with access to the data partner account.

## Instructions

1. Make a copy of this Colab.
2. Fill in the prompts in **Phase 0**.
3. Run your copy of the Colab.

### Step 0. Install and Import Packages

In [ ]:
!pip install --upgrade google-ads-datamanager google-auth-oauthlib

In [ ]:
import datetime
import getpass
from google.ads import datamanager_v1
from google.api_core import exceptions
import google.auth
from google.colab import files
from google.oauth2.credentials import Credentials
from google.protobuf.json_format import MessageToJson

## Phase 0: Provide OAuth Information and Account IDs

### Step 1: Provide credentials information and account IDs

In [ ]:
# @markdown ### Account IDs
# @markdown Provide the advertiser and data partner account IDs below.
# @markdown - **[required]** `login_account_id`: Your Data Partner account.
# @markdown - **[required]** `operating_account_id`: The Google Ads advertiser account that will own the audience you create as part of this workflow.
# @markdown - (*optional*}) `linked_account_id`: If you want to link the `operating_account` directly to your data partner account, leave this blank. If instead you want to link a Google Ads manager account (where the `operating_account` is a child) to your data partner account, set this to the customer ID of the Google Ads manager account.
login_account_id = ""  # @param {type:"string"}
operating_account_id = ""  # @param {type:"string"}
linked_account_id = ""  # @param {type:"string"}

# @markdown *Check this if you already have a Client ID, Client Secret, and Refresh Token for the Advertiser to bypass the gcloud setup:*
use_existing_advertiser_credentials = False  # @param {type:"boolean"}

# @markdown ### Service Account Details (If applicable)
service_account_email = ""  # @param {type:"string"}

# Clean up account IDs by removing hyphens.
login_account_id = login_account_id.replace("-", "")
operating_account_id = operating_account_id.replace("-", "")
linked_account_id = linked_account_id.replace("-", "")

# Sets the linked_account_id to the operating account if linking directly to the operating account.
linked_account_id = linked_account_id or operating_account_id

# Constants.
GOOGLE_ADS = "GOOGLE_ADS"
DATA_PARTNER = "DATA_PARTNER"
CONSENT_GRANTED = "CONSENT_GRANTED"
PARTNER_LINK_SCOPE = "https://www.googleapis.com/auth/datamanager.partnerlink"
DESKTOP_APP_JSON_FILE = "/tmp/oauth_client.json"

### Step 2: Upload OAuth Desktop client JSON for the Advertiser credentials

**IMPORTANT:** Follow the prompts in the command output to complete this step.

In [ ]:
if use_existing_advertiser_credentials:
    print(
        "Skipping OAuth client JSON upload since we're using preexisting Advertiser OAuth credentials."
    )
else:
    # Prompts the user to upload the JSON file
    print(
        "Upload the client JSON file for your Google Cloud Desktop OAuth client:"
    )
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError("Upload exactly one file")

    filename = list(uploaded.keys())[0]
    content = uploaded[filename]

    # Saves the uploaded file.
    with open(DESKTOP_APP_JSON_FILE, "wb") as f:
        f.write(content)

## Phase 1: Advertiser Creates Partner Link

### Step 1. Initialize Advertiser Client

**IMPORTANT:** Follow the prompts in the command output to complete this step.

In [ ]:
class DataManagerAdvertiserSDK:
    def __init__(self, creds):
        self.link_service = datamanager_v1.PartnerLinkServiceClient(credentials=creds)

def initialize_advertiser_client():
    print("Authenticating Advertiser with OAuth...")
    creds = None

    try:
        if use_existing_advertiser_credentials:
            print("Using existing Advertiser OAuth credentials")
            print()
            # Prompts the user for the client ID, client secret, and refresh token.
            print("Enter your preexisting Advertiser OAuth credentials:")
            advertiser_oauth_client_id = input("Client ID: ").strip()
            advertiser_oauth_client_secret = getpass.getpass(
                "Client Secret (input will be hidden): "
            ).strip()
            advertiser_oauth_client_refresh_token = getpass.getpass(
                "Refresh Token (input will be hidden): "
            ).strip()

            # Creates user credentials using the provided values.
            creds = Credentials(
                token=None,
                client_id=advertiser_oauth_client_id,
                client_secret=advertiser_oauth_client_secret,
                refresh_token=advertiser_oauth_client_refresh_token,
                token_uri="https://oauth2.googleapis.com/token",
                scopes=[PARTNER_LINK_SCOPE]
            )
        else:
            # Run the gcloud command
            # Uses the less permissive datamanager.partnerlink scope since the
            # only action the advertiser needs to perform is creating the
            # partner link.
            print("Running gcloud auth to get Advertiser credentials...")
            auth_command = (
                f"gcloud auth application-default login "
                f"--client-id-file='{DESKTOP_APP_JSON_FILE}' "
                f"--scopes={PARTNER_LINK_SCOPE},https://www.googleapis.com/auth/cloud-platform "
                f"--no-browser"
            )
            get_ipython().system(auth_command)

            print()
            print("gcloud auth process finished.")

            creds, project = google.auth.default(scopes=[PARTNER_LINK_SCOPE])

    except Exception as e:
        print(f"An error occurred: {e}")

    return DataManagerAdvertiserSDK(creds)

advertiser_sdk = initialize_advertiser_client()
print("Advertiser SDK Initialized")

### Step 2. Create Partner Link

In [ ]:
print(f"Creating link for Ads {linked_account_id}..\n")

parent_ads = f"accountTypes/GOOGLE_ADS/accounts/{linked_account_id}"

partner_link_data = datamanager_v1.PartnerLink(
    owning_account=datamanager_v1.ProductAccount(
        account_id=linked_account_id, account_type=GOOGLE_ADS
    ),
    partner_account=datamanager_v1.ProductAccount(
        account_id=login_account_id, account_type=DATA_PARTNER
    ),
)

try:
    link_res = advertiser_sdk.link_service.create_partner_link(
        parent=parent_ads, partner_link=partner_link_data
    )
    print(f"Link Created: {link_res.partner_link_id}")
    partner_link_id = link_res.partner_link_id
except exceptions.PermissionDenied as e:
    print(f"Link creation failed due to permission denied error: {e}")
    print(
        "Review the 'Prerequisites' section and verify that the Google Account for advertiser "
        "credentials has the required access level in the advertiser account."
    )
except Exception as e:
    print(f"Link creation failed or exists: {e}")
    print("\nSearching for existing link...")
    search_res = advertiser_sdk.link_service.search_partner_links(
        parent=f"accountTypes/DATA_PARTNER/accounts/{login_account_id}"
    )
    partner_link_id = None
    for link in search_res:
        if link.owning_account.account_id == linked_account_id:
            partner_link_id = link.partner_link_id
            print(f"Found existing Link ID: {partner_link_id}")
            break
    if not partner_link_id:
        print(
            "No existing link found. Review the error details above for more information."
        )

print(
    "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.partnerLinks/create?apix=true"
)
print(f"\nparent: {parent_ads}")
print("\n--- Request JSON Payload ---")
print(MessageToJson(partner_link_data._pb))
print("----------------------------\n")

## Phase 2: Data Partner Ingests Data

### Step 1. Initialize Data Partner Client

In [ ]:
class DataManagerPartnerSDK:
    def __init__(self, creds):
        self.user_list_service = datamanager_v1.UserListServiceClient(credentials=creds)
        self.ingestion_service = datamanager_v1.IngestionServiceClient(credentials=creds)

def initialize_partner_client():
    print("Authenticating Data Partner with Service Account...")
    data_manager_scope = "https://www.googleapis.com/auth/datamanager"
    auth_command = (
        f"gcloud auth application-default login "
        f"--impersonate-service-account={service_account_email} "
        f"--scopes={data_manager_scope},https://www.googleapis.com/auth/cloud-platform "
        f"--no-browser"
    )
    get_ipython().system(auth_command)
    creds, project = google.auth.default(scopes=[data_manager_scope])
    return DataManagerPartnerSDK(creds)

partner_sdk = initialize_partner_client()
print("Data Partner SDK Initialized")

### Step 2. Create User List

In [ ]:
print(f"Creating User List in {operating_account_id}...\n")

parent_userlist = f"accountTypes/GOOGLE_ADS/accounts/{operating_account_id}"

user_list_data = datamanager_v1.UserList(
    display_name=f"Python SDK Audience - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    ingested_user_list_info=datamanager_v1.IngestedUserListInfo(
        upload_key_types=["CONTACT_ID"]
    ),
)

login_account = f"accountTypes/DATA_PARTNER/accounts/{login_account_id}"
linked_account = f"accountTypes/GOOGLE_ADS/accounts/{linked_account_id}"

# Creates a list of header tuples.
headers = [
    ("login-account", login_account),
    ("linked-account", linked_account),
]

destination_id = None
try:
    ulist_res = partner_sdk.user_list_service.create_user_list(
        parent=parent_userlist, user_list=user_list_data, metadata=headers
    )

    destination_id = ulist_res.id

    print(f"User List Created!")
    print(f"Destination ID: {destination_id}")
    print(f"Resource Name: {ulist_res.name}")
    print(f"Display Name: {ulist_res.display_name}")

except exceptions.PermissionDenied as e:
    print(f"User list creation failed due to permission denied error: {e}")
    print(
        "Review the 'Prerequisites' section and verify that the service account has the required "
        "access level in the data partner account."
    )
except Exception as e:
    print(f"Failed to create User List: {e}")

print(
    "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.userLists/create?apix=true"
)
print(f"\nparent: {parent_userlist}")
print(f"login-account: {login_account}")
print(f"linked-account: {linked_account}")
print("\n--- Request JSON Payload ---")
print(MessageToJson(user_list_data._pb))
print("----------------------------\n")

### Step 3. Ingest Data

In [ ]:
ingestion_request_id = None

if not destination_id:
    print(
        "Error: destination_id is not set. Cannot ingest data. Make sure the User List was created successfully in the previous step."
    )
else:
    print(f"Ingesting sample data to User List {destination_id}...")

    destination_obj = datamanager_v1.Destination(
        login_account=datamanager_v1.ProductAccount(
            account_type=DATA_PARTNER, account_id=login_account_id
        ),
        operating_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS, account_id=operating_account_id
        ),
        linked_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS, account_id=linked_account_id
        ),
        product_destination_id=str(destination_id),
    )

    ingest_payload = datamanager_v1.IngestAudienceMembersRequest(
        consent=datamanager_v1.Consent(
            ad_user_data=CONSENT_GRANTED, ad_personalization=CONSENT_GRANTED
        ),
        encoding=datamanager_v1.Encoding.HEX,
        terms_of_service=datamanager_v1.TermsOfService(
            customer_match_terms_of_service_status="ACCEPTED"
        ),
        validate_only=False,
        audience_members=[
            datamanager_v1.AudienceMember(
                composite_data=datamanager_v1.CompositeData(
                    user_data=datamanager_v1.UserData(
                        user_identifiers=[
                            datamanager_v1.UserIdentifier(
                                email_address="223EBDA6F6889B1494551BA902D9D381DAF2F642BAE055888E96343D53E9F9C4"
                            ),
                            datamanager_v1.UserIdentifier(
                                email_address="F1FCDE379F31F4D446B76EE8F34860ECA2288ADC6B6D6C0FDC56D9EEE75A2FA5"
                            ),
                        ]
                    )
                )
            )
        ],
        destinations=[destination_obj],
    )

    try:
        ingest_res = partner_sdk.ingestion_service.ingest_audience_members(
            request=ingest_payload
        )

        ingestion_request_id = ingest_res.request_id
        print(f"Ingestion Submitted!")
        print(f"Request ID: {ingestion_request_id}")

    except Exception as e:
        print(f"Failed to ingest data: {e}")

    print(
        "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/audienceMembers/ingest?apix=true"
    )
    print("\n--- Request JSON Payload ---")
    print(MessageToJson(ingest_payload._pb))
    print("----------------------------\n")

### Step 4. Check Status

In [ ]:
if not ingestion_request_id:
    print("Status check skipped because ingestion request ID is not set.")
else:
    print(f"Checking status for Request ID: {ingestion_request_id}...")

    # Note: Headers aren't needed for this request.
    status_req = datamanager_v1.RetrieveRequestStatusRequest(
        request_id=ingestion_request_id
    )

    try:
        status_res = partner_sdk.ingestion_service.retrieve_request_status(
            request=status_req
        )

        print(f"Successfully retrieved status!")

        for dest_status in status_res.request_status_per_destination:
            dest_id = dest_status.destination.product_destination_id
            current_status = dest_status.request_status.name

            print(f"Status for destination {dest_id}: {current_status}")

    except Exception as e:
        print(f"Failed to retrieve status: {e}")

    print(
        "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/requestStatus/retrieve?apix=true"
    )
    print("\n--- Request JSON Payload ---")
    print(MessageToJson(status_req._pb))
    print("----------------------------")